# Sentinel-2 data selection and retrieval 

In [1]:
from datetime import datetime
from pathlib import Path
from collections import defaultdict

import requests
import os
import json
import zipfile
import numpy as np
import glob
import matplotlib.pyplot as plt

import zipfile
import rasterio
from rasterio.plot import show
from rasterio.enums import Resampling

In [2]:
# Bounding box coordinates [min_lon, min_lat, max_lon, max_lat]
# Middle of Sweden (Focus Dalarna but covering Uppsala to Umeå), Sweden
min_lon, min_lat = 12.5, 60
max_lon, max_lat = 17.5, 63.5

# Create WKT POLYGON for API query
roi_polygon = f"POLYGON(({min_lon} {min_lat},{max_lon} {min_lat},{max_lon} {max_lat},{min_lon} {max_lat},{min_lon} {min_lat}))"

# Define date range
start_date = '2018-03-01T00:00:00.000Z'
end_date = '2018-10-31T23:59:59.999Z'

# Maximum cloud cover percentage (30% to get good data availability)
max_cloud_cover = 30

In [3]:

COPERNICUS_CLIENT_ID = os.getenv('COPERNICUS_CLIENT_ID', 'sh-c5dcc309-63e8-491b-8c97-47925cbe91ea')
COPERNICUS_CLIENT_SECRET = os.getenv('COPERNICUS_CLIENT_SECRET', 'uZ0NKF3IlVgFdGQUbSOYlLDrdbjudxtL')

COPERNICUS_USERNAME = "acc6@hi.is"
COPERNICUS_PASSWORD = "CutiePatootie5!"

AUTH_URL = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
SEARCH_URL = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
DOWNLOAD_URL = "https://zipper.dataspace.copernicus.eu/odata/v1/Products"

In [4]:
user = "team1"
selected_tile = "T33VWH"
download_dir = f"/p/scratch/training2600/{user}/data"
results_dir = os.path.join(os.environ['PROJECT_training2600'], f"{user}/results")

## Function definitions

In [5]:
def get_access_token(client_id, client_secret):
    """
    Get OAuth2 access token from Copernicus Dataspace using Client Credentials flow.
    
    This is the recommended method for server-to-server authentication and HPC jobs.
    
    Parameters:
    -----------
    client_id : str
        OAuth2 Client ID (starts with 'sh-')
    client_secret : str
        OAuth2 Client Secret
    
    Returns:
    --------
    str : Access token if successful, None otherwise
    """
    data = {
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret,
    }
    
    try:
        response = requests.post(AUTH_URL, data=data, timeout=30)
        response.raise_for_status()
        token_data = response.json()
        
        access_token = token_data["access_token"]
        expires_in = token_data.get("expires_in", 3600)
        
        print(f"✓ Token obtained (valid for {expires_in//60} minutes)")
        return access_token
        
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 401:
            print("❌ Authentication failed: Invalid credentials")
        else:
            print(f"❌ HTTP {e.response.status_code}: {e.response.text}")
        return None
        
    except Exception as e:
        print(f"❌ Authentication failed: {e}")
        return None
    
def search_sentinel2(start_date, end_date, roi_polygon, max_cloud_cover):
    """
    Search for Sentinel-2 L2A products in Copernicus Dataspace
    """
    # Build OData filter query
    filters = [
        f"Collection/Name eq 'SENTINEL-2'",
        f"Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' and att/OData.CSC.StringAttribute/Value eq 'S2MSI2A')",
        f"ContentDate/Start gt {start_date}",
        f"ContentDate/Start lt {end_date}",
        f"OData.CSC.Intersects(area=geography'SRID=4326;{roi_polygon}')",
        f"Attributes/OData.CSC.DoubleAttribute/any(att:att/Name eq 'cloudCover' and att/OData.CSC.DoubleAttribute/Value lt {max_cloud_cover})"
    ]
    
    filter_query = " and ".join(filters)
    
    params = {
        "$filter": filter_query,
        "$orderby": "ContentDate/Start asc",
        "$top": 1000  # Increased to get full date range across all tiles
    }
    
    try:
        response = requests.get(SEARCH_URL, params=params, timeout=60)
        response.raise_for_status()
        results = response.json()
        return results.get('value', [])
    except Exception as e:
        print(f"❌ Search failed: {e}")
        return []

def download_product(product, output_dir, access_token):
    """
    Download a Sentinel-2 product from Copernicus Dataspace
    
    Parameters:
    -----------
    product : dict
        Product metadata from search results
    output_dir : str
        Directory to save downloaded file
    access_token : str
        OAuth2 access token
    
    Returns:
    --------
    str : Path to downloaded file, or None if failed
    """
    product_id = product['Id']
    product_name = product['Name']
    
    # Create download directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Output file path
    output_file = os.path.join(output_dir, f"{product_name}.zip")
    
    # Check if already downloaded
    if os.path.exists(output_file):
        print(f"⚠ File already exists: {product_name}.zip")
        return output_file
    
    # Build download URL
    download_url = f"{DOWNLOAD_URL}({product_id})/$value"
    
    headers = {"Authorization": f"Bearer {access_token}"}
    
    try:
        print(f"Downloading: {product_name}")
        print(f"  Size: {product['ContentLength'] / (1024**3):.2f} GB")
        
        # Stream download with progress
        with requests.get(download_url, headers=headers, stream=True, timeout=300) as response:
            response.raise_for_status()
            
            total_size = int(response.headers.get('content-length', 0))
            block_size = 8192
            downloaded = 0
            
            with open(output_file, 'wb') as f:
                for chunk in response.iter_content(chunk_size=block_size):
                    if chunk:
                        f.write(chunk)
                        downloaded += len(chunk)
                        
                        # Print progress every 100 MB
                        if downloaded % (100 * 1024 * 1024) < block_size:
                            progress = (downloaded / total_size) * 100 if total_size > 0 else 0
                            print(f"  Progress: {progress:.1f}% ({downloaded / (1024**3):.2f} GB)")
        
        print(f"✓ Download complete: {output_file}")
        return output_file
        
    except Exception as e:
        print(f"❌ Download failed: {e}")
        # Clean up partial download
        if os.path.exists(output_file):
            os.remove(output_file)
        return None

def create_authenticated_session(username, password):
    """
    Create a session with authentication for downloads
    """
    session = requests.Session()
    
    # Get access token
    data = {
        "client_id": "cdse-public",
        "username": username,
        "password": password,
        "grant_type": "password",
    }
    
    response = session.post(AUTH_URL, data=data, timeout=30)
    response.raise_for_status()
    
    token = response.json()["access_token"]
    session.headers.update({"Authorization": f"Bearer {token}"})
    
    return session

def download_with_session(product, output_dir, session):
    """
    Download using authenticated session
    """
    product_id = product['Id']
    product_name = product['Name']
    output_file = os.path.join(output_dir, f"{product_name}.zip")
    
    if os.path.exists(output_file):
        print(f"⚠ File already exists: {product_name}.zip")
        return output_file
    
    download_url = f"{DOWNLOAD_URL}({product_id})/$value"
    
    try:
        print(f"Downloading: {product_name}")
        print(f"  Size: {product['ContentLength'] / (1024**3):.2f} GB")
        
        with session.get(download_url, stream=True, timeout=300) as response:
            response.raise_for_status()
            
            total_size = int(response.headers.get('content-length', 0))
            downloaded = 0
            
            os.makedirs(output_dir, exist_ok=True)
            
            with open(output_file, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
                        downloaded += len(chunk)
                        
                        if downloaded % (100 * 1024 * 1024) < 8192:
                            progress = (downloaded / total_size) * 100 if total_size > 0 else 0
                            print(f"  Progress: {progress:.1f}%")
        
        print(f"✓ Download complete: {output_file}")
        return output_file
        
    except Exception as e:
        print(f"❌ Download failed: {e}")
        if os.path.exists(output_file):
            os.remove(output_file)
        return None

def extract_and_visualize_sentinel2(zip_path, output_dir=None):
    """
    Extract a Sentinel-2 ZIP file and create an RGB visualization.
    
    Parameters:
    -----------
    zip_path : str
        Path to the Sentinel-2 ZIP file
    output_dir : str, optional
        Directory to extract to. If None, extracts to same directory as ZIP.
    
    Returns:
    --------
    str : Path to extracted SAFE directory
    """    
    if output_dir is None:
        output_dir = os.path.dirname(zip_path)
    
    # Extract ZIP file
    print(f"Extracting: {os.path.basename(zip_path)}")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        # Get the SAFE directory name (first item in the archive)
        safe_dir = zip_ref.namelist()[0].split('/')[0]
        safe_path = os.path.join(output_dir, safe_dir)
        
        if os.path.exists(safe_path):
            print(f"⚠ Already extracted: {safe_dir}")
        else:
            zip_ref.extractall(output_dir)
            print(f"✓ Extracted to: {safe_path}")
    
    return safe_path


def visualize_sentinel2_rgb(safe_path, figsize=(12, 12), is_show_plot=False):
    """
    Create an RGB visualization from Sentinel-2 SAFE directory.
    
    Uses bands B04 (Red), B03 (Green), B02 (Blue) at 10m resolution.
    
    Parameters:
    -----------
    safe_path : str
        Path to the extracted .SAFE directory
    figsize : tuple
        Figure size for the plot
    """
    # Find the 10m resolution bands (B02, B03, B04)
    # Path pattern: .SAFE/GRANULE/*/IMG_DATA/R10m/*_B0X_10m.jp2
    granule_path = os.path.join(safe_path, 'GRANULE')
    
    if not os.path.exists(granule_path):
        print(f"❌ GRANULE directory not found in {safe_path}")
        return
    
    # Get the tile subdirectory
    tile_dirs = [d for d in os.listdir(granule_path) if os.path.isdir(os.path.join(granule_path, d))]
    if not tile_dirs:
        print("❌ No tile directories found")
        return
    
    tile_dir = os.path.join(granule_path, tile_dirs[0])
    
    # Try R10m directory first (newer format), then IMG_DATA (older format)
    r10m_path = os.path.join(tile_dir, 'IMG_DATA', 'R10m')
    img_data_path = os.path.join(tile_dir, 'IMG_DATA')
    
    if os.path.exists(r10m_path):
        band_dir = r10m_path
        band_pattern = '*_B0{}_10m.jp2'
    else:
        band_dir = img_data_path
        band_pattern = '*_B0{}.jp2'
    
    # Find band files
    bands = {}
    for band_num in ['2', '3', '4']:
        pattern = os.path.join(band_dir, band_pattern.format(band_num))
        matches = glob.glob(pattern)
        if matches:
            bands[f'B0{band_num}'] = matches[0]
        else:
            # Try alternative pattern for older format
            alt_pattern = os.path.join(band_dir, f'*B0{band_num}*.jp2')
            alt_matches = glob.glob(alt_pattern)
            if alt_matches:
                bands[f'B0{band_num}'] = alt_matches[0]
    
    if len(bands) < 3:
        print(f"❌ Could not find all RGB bands. Found: {list(bands.keys())}")
        print(f"   Searched in: {band_dir}")
        return
    
    print(f"✓ Found bands: {list(bands.keys())}")
    
    # Read bands with downsampling for visualization (full resolution can be huge)
    downsample_factor = 10  # Read at 1/10 resolution for faster display
    
    rgb_bands = []
    for band_name in ['B04', 'B03', 'B02']:  # RGB order
        with rasterio.open(bands[band_name]) as src:
            # Calculate new dimensions
            new_height = src.height // downsample_factor
            new_width = src.width // downsample_factor
            
            # Read with resampling
            band_data = src.read(
                1,
                out_shape=(new_height, new_width),
                resampling=Resampling.average
            )
            rgb_bands.append(band_data)
    
    # Stack into RGB array
    rgb = np.stack(rgb_bands, axis=-1)
    
    # Normalize for visualization (typical Sentinel-2 values range 0-10000)
    # Use percentile-based stretching for better visualization
    p2, p98 = np.percentile(rgb[rgb > 0], (2, 98))
    rgb_normalized = np.clip((rgb - p2) / (p98 - p2), 0, 1)
    
    # Create visualization
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # RGB composite
    axes[0].imshow(rgb_normalized)
    axes[0].set_title('Sentinel-2 RGB Composite (B4-B3-B2)', fontsize=12)
    axes[0].axis('off')
    
    # False color (NIR-Red-Green) if available
    # For now, show a histogram of the data
    axes[1].hist(rgb[:,:,0].flatten()[::100], bins=50, alpha=0.7, label='Red (B04)', color='red')
    axes[1].hist(rgb[:,:,1].flatten()[::100], bins=50, alpha=0.7, label='Green (B03)', color='green')
    axes[1].hist(rgb[:,:,2].flatten()[::100], bins=50, alpha=0.7, label='Blue (B02)', color='blue')
    axes[1].set_xlabel('Reflectance Value')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Band Value Distribution', fontsize=12)
    axes[1].legend()
    axes[1].set_xlim(0, 5000)
    
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, "vis_S2.png"))

    if is_show_plot:
        plt.show()
    else:
        plt.close()
    
    # Print some metadata
    with rasterio.open(bands['B04']) as src:
        print(f"\nImage Metadata:")
        print(f"  Original size: {src.width} x {src.height} pixels")
        print(f"  Resolution: {src.res[0]}m x {src.res[1]}m")
        print(f"  CRS: {src.crs}")
        print(f"  Bounds: {src.bounds}")
    
    return rgb_normalized

def is_valid_zipfile(filepath):
    """Check if a file is a valid ZIP file."""
    try:
        with zipfile.ZipFile(filepath, 'r') as zf:
            return zf.testzip() is None
    except (zipfile.BadZipFile, Exception):
        return False

## Copernicus Open Access Hub Authentication

In [6]:
access_token = get_access_token(COPERNICUS_CLIENT_ID, COPERNICUS_CLIENT_SECRET)

if access_token:
    print("✓ Successfully authenticated")
    headers = {"Authorization": f"Bearer {access_token}"}
else:
    print("❌ Authentication failed.")

✓ Token obtained (valid for 30 minutes)
✓ Successfully authenticated


## Search Sentinel-2 Collection

In [7]:
products = search_sentinel2(start_date, end_date, roi_polygon, max_cloud_cover)

print(f"\n✓ Found {len(products)} Sentinel-2 L2A products")
print(f"✓ All products have <{max_cloud_cover}% cloud cover (filtered server-side)")
print(f"\nThese products span multiple MGRS tiles over your region.")
print(f"Select one MGRS tile and download ~4 acquisitions.\n")
print(f"First 5 products:")
for i, product in enumerate(products[:5]):
    name = product.get('Name', 'Unknown')
    date = product.get('ContentDate', {}).get('Start', 'N/A')[:10]
    size = product.get('ContentLength', 0) / (1024**3)
    
    print(f"  {i+1}. {name}")
    print(f"     Date: {date}, Size: {size:.2f} GB")


✓ Found 1000 Sentinel-2 L2A products
✓ All products have <30% cloud cover (filtered server-side)

These products span multiple MGRS tiles over your region.
Select one MGRS tile and download ~4 acquisitions.

First 5 products:
  1. S2B_MSIL2A_20180301T103019_N0500_R108_T33VUK_20230731T095708.SAFE
     Date: 2018-03-01, Size: 0.20 GB
  2. S2B_MSIL2A_20180301T103019_N0500_R108_T32VPQ_20230731T095708.SAFE
     Date: 2018-03-01, Size: 0.12 GB
  3. S2B_MSIL2A_20180301T103019_N0500_R108_T33VWJ_20230731T095708.SAFE
     Date: 2018-03-01, Size: 1.18 GB
  4. S2B_MSIL2A_20180301T103019_N0500_R108_T33VXL_20230731T095708.SAFE
     Date: 2018-03-01, Size: 1.17 GB
  5. S2B_MSIL2A_20180301T103019_N0500_R108_T33VVJ_20230731T095708.SAFE
     Date: 2018-03-01, Size: 1.16 GB


## Group Products by MGRS Tile

In [8]:
tiles = defaultdict(list)
for product in products:
    product_name = product.get('Name', '')
    tile_id = product_name.split('_')[5] if len(product_name.split('_')) > 5 else 'Unknown'
    tiles[tile_id].append(product)

# Display available tiles and their acquisition counts
print("Available MGRS Tiles and Acquisition Counts:")
print("=" * 50)
for tile_id, tile_products in sorted(tiles.items(), key=lambda x: len(x[1]), reverse=True):
    print(f"\nTile {tile_id}: {len(tile_products)} acquisitions")
    for i, product in enumerate(tile_products[:5]):  # Show first 5
        date = product.get('ContentDate', {}).get('Start', 'N/A')[:10]
        size = product.get('ContentLength', 0) / (1024**3)
        print(f"  {i+1}. {date} - {size:.2f} GB")
    if len(tile_products) > 5:
        print(f"  ... and {len(tile_products) - 5} more")

print("\n" + "=" * 50)
print(f"\nRecommendation: Choose a tile with 4+ acquisitions for training data diversity.")

Available MGRS Tiles and Acquisition Counts:

Tile T33VWH: 58 acquisitions
  1. 2018-03-14 - 0.10 GB
  2. 2018-03-15 - 0.17 GB
  3. 2018-03-16 - 1.16 GB
  4. 2018-03-18 - 1.18 GB
  5. 2018-03-20 - 0.18 GB
  ... and 53 more

Tile T34VCN: 48 acquisitions
  1. 2018-03-18 - 0.95 GB
  2. 2018-03-20 - 0.94 GB
  3. 2018-03-21 - 0.12 GB
  4. 2018-03-25 - 0.84 GB
  5. 2018-03-26 - 0.10 GB
  ... and 43 more

Tile T34VCP: 46 acquisitions
  1. 2018-03-01 - 0.51 GB
  2. 2018-03-11 - 0.61 GB
  3. 2018-03-20 - 0.79 GB
  4. 2018-03-25 - 0.70 GB
  5. 2018-03-26 - 0.46 GB
  ... and 41 more

Tile T33VXJ: 46 acquisitions
  1. 2018-03-01 - 0.73 GB
  2. 2018-03-11 - 0.82 GB
  3. 2018-03-20 - 0.69 GB
  4. 2018-03-21 - 0.74 GB
  5. 2018-03-25 - 0.60 GB
  ... and 41 more

Tile T33VXH: 45 acquisitions
  1. 2018-03-18 - 1.08 GB
  2. 2018-03-20 - 1.02 GB
  3. 2018-03-21 - 0.43 GB
  4. 2018-03-25 - 0.98 GB
  5. 2018-03-28 - 1.01 GB
  ... and 40 more

Tile T33VWG: 43 acquisitions
  1. 2018-03-20 - 0.53 GB
  2. 2018

In [9]:
tile_products = tiles[selected_tile]
num_acquisitions = len(tile_products)

print(f"Selected MGRS Tile: {selected_tile}")
print(f"Total acquisitions available: {num_acquisitions}")

# Print ALL acquisitions
print("\nAll acquisitions:")
print("=" * 70)
for i, product in enumerate(tile_products, start=1):
    name = product.get('Name', 'Unknown')
    date = product.get('ContentDate', {}).get('Start', 'N/A')[:10]
    size = product.get('ContentLength', 0) / (1024**3)
    print(f"{i:3d}. {date} - {size:.2f} GB")
    print(f"     {name}")
print("=" * 70)

indices = [3,15,34,56] # List of indices for picked acquisitions
selected_products = [tile_products[i] for i in indices]

print("List of chosen acquisitions")
print("=" * 70)
for i, (idx, product) in enumerate(zip(indices, selected_products)):
    name = product.get('Name', 'Unknown')
    date = product.get('ContentDate', {}).get('Start', 'N/A')[:10]
    size = product.get('ContentLength', 0) / (1024**3)
    print(f"  {i+1}. [{idx+1}/{num_acquisitions}] {date} - {size:.2f} GB")
    print(f"      {name}")
print("=" * 70)

Selected MGRS Tile: T33VWH
Total acquisitions available: 58

All acquisitions:
  1. 2018-03-14 - 0.10 GB
     S2B_MSIL2A_20180314T104019_N0500_R008_T33VWH_20230912T132719.SAFE
  2. 2018-03-15 - 0.17 GB
     S2B_MSIL2A_20180315T101019_N0500_R022_T33VWH_20230908T210616.SAFE
  3. 2018-03-16 - 1.16 GB
     S2A_MSIL2A_20180316T103021_N0500_R108_T33VWH_20230727T202428.SAFE
  4. 2018-03-18 - 1.18 GB
     S2B_MSIL2A_20180318T102019_N0500_R065_T33VWH_20230903T071056.SAFE
  5. 2018-03-20 - 0.18 GB
     S2A_MSIL2A_20180320T101021_N0500_R022_T33VWH_20230903T133606.SAFE
  6. 2018-03-25 - 0.17 GB
     S2B_MSIL2A_20180325T101019_N0500_R022_T33VWH_20230828T070713.SAFE
  7. 2018-03-28 - 1.16 GB
     S2B_MSIL2A_20180328T102019_N0500_R065_T33VWH_20230829T155330.SAFE
  8. 2018-03-30 - 0.18 GB
     S2A_MSIL2A_20180330T101021_N0500_R022_T33VWH_20230828T061911.SAFE
  9. 2018-04-02 - 1.16 GB
     S2A_MSIL2A_20180402T102021_N0500_R065_T33VWH_20230726T115942.SAFE
 10. 2018-04-12 - 1.18 GB
     S2A_MSIL2A_201804

## Download Sentinel-2 Data

In [10]:
session = create_authenticated_session(COPERNICUS_USERNAME, COPERNICUS_PASSWORD)

print(f"Downloading {len(selected_products)} evenly-spaced acquisitions from tile {selected_tile}...")
print("=" * 60)

for i, product in enumerate(selected_products):
    date = product.get('ContentDate', {}).get('Start', 'N/A')[:10]
    print(f"\n--- Downloading {i+1}/{len(selected_products)} ({date}) ---")
    download_with_session(product, download_dir, session)

print("\n" + "=" * 60)
print(f"✓ Downloaded {len(selected_products)} acquisitions from tile {selected_tile}")
print(f"✓ Acquisitions are evenly spaced across the time range")
print(f"✓ All files saved to: {download_dir}")


--- Downloading 1/4 (2018-03-18) ---
Downloading: S2B_MSIL2A_20180318T102019_N0500_R065_T33VWH_20230903T071056.SAFE
  Size: 1.18 GB
  Progress: 8.3%
  Progress: 16.5%
  Progress: 24.8%
  Progress: 33.1%
  Progress: 41.3%
  Progress: 49.6%
  Progress: 57.9%
  Progress: 66.1%
  Progress: 74.4%
  Progress: 82.7%
  Progress: 90.9%
  Progress: 99.2%
✓ Download complete: /p/scratch/training2600/team1/data/S2B_MSIL2A_20180318T102019_N0500_R065_T33VWH_20230903T071056.SAFE.zip

--- Downloading 2/4 (2018-04-22) ---
Downloading: S2A_MSIL2A_20180422T102031_N0500_R065_T33VWH_20230915T161912.SAFE
  Size: 1.18 GB
  Progress: 8.2%
  Progress: 16.5%
  Progress: 24.7%
  Progress: 33.0%
  Progress: 57.7%
  Progress: 66.0%
  Progress: 74.2%
  Progress: 82.5%
  Progress: 90.7%
  Progress: 99.0%
✓ Download complete: /p/scratch/training2600/team1/data/S2A_MSIL2A_20180422T102031_N0500_R065_T33VWH_20230915T161912.SAFE.zip

--- Downloading 3/4 (2018-06-04) ---
Downloading: S2A_MSIL2A_20180604T103021_N0500_R108

In [12]:
first_product = selected_products[0] # Adjust to decide which product for vizualise
product_name = first_product.get('Name', '')
base_name = product_name.rstrip('.SAFE') if product_name.endswith('.SAFE') else product_name

possible_paths = [
    os.path.join(download_dir, f"{product_name}"),           # Direct name (if it's a directory)
    os.path.join(download_dir, f"{product_name}.SAFE"),      # With .SAFE suffix
    os.path.join(download_dir, f"{base_name}.SAFE"),         # Base name with .SAFE
    os.path.join(download_dir, f"{product_name}.zip"),       # With .zip suffix  
    os.path.join(download_dir, f"{base_name}.zip"),          # Base name with .zip
]

safe_path = None
zip_path = None

# Find the first existing path
for path in possible_paths:
    if os.path.exists(path):
        if os.path.isdir(path):
            # It's a directory (SAFE format)
            safe_path = path
            print(f"✓ Found SAFE directory: {os.path.basename(path)}")
            break
        elif path.endswith('.zip') and is_valid_zipfile(path):
            # It's a valid ZIP file
            zip_path = path
            print(f"✓ Found valid ZIP file: {os.path.basename(path)}")
            break
        elif path.endswith('.zip'):
            # File exists but is not a valid ZIP - might be misnamed SAFE dir
            print(f"⚠ Found {os.path.basename(path)} but it's not a valid ZIP file")
            print("  Checking if it might be a SAFE directory saved with wrong extension...")
            # Check if there's a SAFE directory with similar name
            continue

# If we found a ZIP file, extract it
if zip_path and not safe_path:
    print(f"\nVisualizing: {os.path.basename(zip_path)}")
    print("=" * 60)
    safe_path = extract_and_visualize_sentinel2(zip_path)

# Visualize if we have a SAFE path
if safe_path and os.path.isdir(safe_path):
    print(f"\nVisualizing: {os.path.basename(safe_path)}")
    print("=" * 60)
    
    # Create RGB visualization directly from SAFE directory
    print("\nCreating RGB visualization...")
    visualize_sentinel2_rgb(safe_path, is_show_plot=True)
    
    print("\n" + "=" * 60)
    print("✓ Visualization complete!")
    print("  - RGB composite shows the natural color view")
    print("  - Histogram shows the distribution of reflectance values")
else:
    # Last resort: try to find any .SAFE directory in download_dir
    safe_dirs = glob.glob(os.path.join(download_dir, "*.SAFE"))
    if safe_dirs:
        safe_path = safe_dirs[0]
        print(f"\nFound SAFE directory: {os.path.basename(safe_path)}")
        print("=" * 60)
        
        print("\nCreating RGB visualization...")
        visualize_sentinel2_rgb(safe_path, is_show_plot=True)
        
        print("\n" + "=" * 60)
        print("✓ Visualization complete!")
    else:
        print(f"\n❌ No valid Sentinel-2 data found in: {download_dir}")
        print(f"\nSearched for:")
        for p in possible_paths[:3]:
            print(f"  - {p}")
        print("\nTip: Check what files exist in your download directory:")
        print(f"  ls -la {download_dir}")

✓ Found SAFE directory: S2B_MSIL2A_20180318T102019_N0500_R065_T33VWH_20230903T071056.SAFE

Visualizing: S2B_MSIL2A_20180318T102019_N0500_R065_T33VWH_20230903T071056.SAFE

Creating RGB visualization...
✓ Found bands: ['B02', 'B03', 'B04']

Image Metadata:
  Original size: 10980 x 10980 pixels
  Resolution: 10.0m x 10.0m
  CRS: EPSG:32633
  Bounds: BoundingBox(left=499980.0, bottom=6690240.0, right=609780.0, top=6800040.0)

✓ Visualization complete!
  - RGB composite shows the natural color view
  - Histogram shows the distribution of reflectance values
